# Retrieve Contextualized Word Embeddings from BERT base

In [ ]:
# Packages
import pandas as pd
import json
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import gc
from pathlib import Path
import time
from datetime import datetime

## Import Mentions Data
Data uploaded from `data/mentions/narrative_mentions.jsonl`

In [1]:
#file = "../data/mentions/narrative_mentions.jsonl"
file = '/kaggle/input/datasets/lianestrauch/narrative-mentions-jsonl/narrative_mentions.jsonl'

records = []

with open(file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            print(f"Skipping malformed line in {file}")

mentions_df = pd.DataFrame(records)

print(mentions_df.shape)
print(mentions_df.columns)

(537645, 15)
Index(['paperId', 'doi', 'oa_id', 'text_type', 'text_cleaned', 'match_type',
       'matched_seq', 'matched_char_start', 'matched_char_end', 'sentence_pos',
       'pos', 'id', 'context_token_ids', 'context_n_tokens',
       'matched_token_indices'],
      dtype='object')


## Set Up and Testing GPUs

In [2]:
!nvidia-smi

Mon Aug 24 18:38:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Check GPU count
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPUs available: {torch.cuda.device_count()}")  # Should print 2

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Send model to main GPU first
model = model.to(device)

# Wrap model to use ALL available GPUs
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

# print diagnostics
model.eval()

# Process data
texts = ["First test sentence.", "Second test sentence."]
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)

# get the embeddings
with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

GPUs available: 2


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 6, 768])


## Update the get_embeddings Function

In [4]:
def get_token_embeddings_batched(sentences, target_indices_list, tokenizer, model, device):
    """
    Extract the last four hidden-state embeddings for selected tokens from a
    batch of sentences.

    The sentences are tokenized as a padded batch and passed through the model
    with hidden-state outputs enabled. For each sentence, the function extracts
    the embeddings from the final four hidden layers for the requested token
    position(s).

    Parameters
    ----------
    sentences : list of str
        Input sentences to process in a list

    target_indices_list : list of list or tuple of int
        Token indices to extract for each sentence. The outer list must have
        the same length as `sentences`. If a single index is provided, the
        embedding for that token is extracted. If multiple indices are
        provided, the function extracts the contiguous token span from the
        first index through the last index (inclusive).

        For example:
        [[2], [1, 2]]

        Note that indices refer to the tokenized input sequence, including any
        special tokens added by the tokenizer (e.g., [CLS] and [SEP] for BERT).

    tokenizer : transformers.PreTrainedTokenizer
        Hugging Face tokenizer used to tokenize `sentences`.

    model : transformers.PreTrainedModel
        Hugging Face transformer model used to generate hidden states. The
        model is expected to support `output_hidden_states=True`.

    device : torch.device or str
        Device on which tokenization tensors and the model inputs are placed,
        e.g. `"cuda"` or `"cpu"`.

    Returns
    -------
    list of numpy.ndarray
        A list containing one NumPy array per sentence.

        For a single target index, the corresponding array has shape:
        (4, hidden_size)

        For multiple target indices, the corresponding array has shape:
        (4, num_selected_tokens, hidden_size)

        The four embedding dimensions correspond to the model's final four
        hidden layers, ordered from the fourth-to-last layer to the final layer.
        The arrays are returned as float32 NumPy arrays on CPU memory.

    Notes
    -----
    Tokenization uses padding and truncation. Therefore, target indices must
    refer to positions that remain within the tokenizer's maximum sequence
    length after truncation.

    For BERT-base, the hidden size is 768 and the model provides 13 hidden-state
    tensors: the initial embedding output followed by the outputs of the
    12 transformer layers. The final four layers are selected here.

    CUDA tensors are explicitly deleted and the CUDA cache is cleared after
    processing to reduce GPU memory usage, which can be useful in notebook
    environments such as Kaggle.
"""
    # 1. Tokenize as a batch with padding enabled
    # Truncation is fine as long as target_indices aren't past max_length (e.g. 512)
    inputs = tokenizer(
        sentences, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    ).to(device)

    # 2. Forward pass on GPU
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # hidden_states tuple length: 13 (for bert-base: embeddings + 12 layers)
    # Stack the last 4 layers -> shape: (4, batch_size, seq_len, 768)
    layers = torch.stack(outputs.hidden_states[-4:])

    batch_embeddings = []

    # 3. Extract requested indices for each sentence in the batch
    for batch_idx, indices in enumerate(target_indices_list):
        if len(indices) == 1:
            # Shape: (4, 768) -> (4th-to-last, ..., last layer) for token at indices[0]
            emb = layers[:, batch_idx, indices[0]]
        else:
            # Shape: (4, num_tokens_selected, 768)
            emb = layers[:, batch_idx, indices[0]: indices[-1] + 1]
        
        # Move back to CPU memory and convert to Float32 NumPy array
        batch_embeddings.append(emb.to(torch.float32).cpu().numpy())

    # --- CRITICAL: Delete tensors and clear CUDA cache ---
    del inputs, outputs, layers
    torch.cuda.empty_cache()

    return batch_embeddings

## Create a Wrapper for the get_embeddings Function

In [5]:
# garbage collection to avoid overflowing GPU

gc.collect()
torch.cuda.empty_cache()
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

Allocated memory: 448.68 MB


In [6]:
# Setup the paths for input and output
embeddings_path = Path("bert-base")
embeddings_path.mkdir(parents=True, exist_ok=True)
progress_path = embeddings_path / "progress.json"

# Load the current progress if it exists
if progress_path.is_file():
    with open(progress_path, "r") as f:
        progress = json.load(f)

    # check to what extent the data has been processed - create to_process_df
    to_process_df = mentions_df[mentions_df["id"].isin(progress["to_process"])] # TODO: make this more robust! (e.g. what if the key does not exist?)
    print(f"1: {len(to_process_df)}")
    # run a quick sum check
    print(f"All papers match: {len(mentions_df) == progress['all_ids']}")
    print(f"Papers to process match: {len(to_process_df) == progress['to_process_n']}")
    ids_done_prior = progress['ids_processed_n'] # for the chunks_naming

    # GO TO PROCESS THE DATA

else:
    # create a progress document and fill in the first set of data
    progress = {}
    progress["all_ids"] = len(mentions_df)
    progress["ids_processed_n"] = 0
    ids_done_prior = progress['ids_processed_n'] #for the chunks naming
    
    # Filter - context >512
    large_context_ids = list(mentions_df[mentions_df.context_n_tokens > 512]["id"])
    progress["ids_rm_len_n"] = len(large_context_ids)
    progress["ids_rm_len"] = large_context_ids
    to_process_df = mentions_df.drop(mentions_df[mentions_df.context_n_tokens > 512].index)
    print(f"2: {len(to_process_df)}")

    # Filter - keep only "narrative" and "narratives"
    non_conservative_match = to_process_df[~to_process_df["matched_seq"].isin(["narrative", "narratives"])]["id"].tolist()
    progress["ids_rm_match_n"] = len(non_conservative_match)
    progress["ids_rm_match"] = non_conservative_match
    to_process_df = to_process_df.drop(to_process_df[to_process_df.id.isin(non_conservative_match)].index)
    print(f"3: {len(to_process_df)}")


    # store the to_process_df info
    progress["to_process_n"] = len(to_process_df)
    progress["to_process"] = to_process_df.id.to_list()

    with open(progress_path, "w") as f:
        json.dump(progress, f, indent=4)

############### PROCESS THE DATA ###################################################
# Work with to_process_df
# index into the (4, ...) stack: 0=4th-to-last, 1=3rd-to-last, 2=2nd-to-last, 3=last
LAYER_COLS = {
    "layer_last": 3,
    "layer_2nd_last": 2,
    "layer_3rd_last": 1,
    "layer_4th_last": 0,
}
to_process_df = to_process_df.reset_index(drop=True)

# Create folder to store the retrieved embeddings
Path(embeddings_path / "chunks").mkdir(parents=True, exist_ok=True)

# Set up time logging
progress["start_time"] = datetime.now().isoformat()
start_ts = time.monotonic()  # for computing elapsed duration cheaply

BATCH_SIZE = 32  # Power of 2 for T4 GPUs

# Prepare records array
records = []
total_rows = len(to_process_df)

# Convert DataFrame columns to lists for fast slicing
all_texts = to_process_df["text_cleaned"].tolist()
all_indices = to_process_df["matched_token_indices"].tolist()
all_ids = to_process_df["id"].tolist()

# Process in batch strides
for start_idx in range(0, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    
    # Slice the current batch
    batch_texts = all_texts[start_idx:end_idx]
    batch_indices = all_indices[start_idx:end_idx]
    batch_ids = all_ids[start_idx:end_idx]

    # Get embeddings for the batch on GPU
    # Returns a list of NumPy arrays (one array per sentence in batch)
    batch_embeddings = get_token_embeddings_batched(
        sentences=batch_texts,
        target_indices_list=batch_indices,
        tokenizer=tokenizer,
        model=model,
        device=device
    )

    # Process results from this batch
    for row_id, embeddings in zip(batch_ids, batch_embeddings):
        record = {"id": row_id}
        
        for col_name, layer_idx in LAYER_COLS.items():
            layer_emb = embeddings[layer_idx]
            
            # Fix for non-conservative matches (multi-token spans):
            # layer_emb shape is (num_tokens, 768) if multi-token, or (768,) if single token.
            if layer_emb.ndim > 1:
                layer_emb = layer_emb.mean(axis=0)  # Average token span to 1D vector
                
            record[col_name] = layer_emb.tolist()
            
        records.append(record)

    # Logging progress
    processed_count = len(records)
    if (start_idx + BATCH_SIZE) % 1000 < BATCH_SIZE:
        print(f"Processed: {min(start_idx + BATCH_SIZE, total_rows):,} / {total_rows:,}")

    # Checkpoint every 4,000 items
    if len(records) >= 4000:
        chunk_df = pd.DataFrame(records)
        processed_ids = {r["id"] for r in records}
        records = []  # Reset buffer

        # Save parquet chunk
        chunk_file = embeddings_path / "chunks" / f"chunk_{start_idx + BATCH_SIZE + ids_done_prior}.parquet"
        chunk_df.to_parquet(chunk_file, index=False)

        print(f"--- Saved Checkpoint: {chunk_file.name} ---")

        # Fast update of progress file using set operations
        progress["ids_processed_n"] += len(processed_ids)
        progress["to_process"] = list(set(progress["to_process"]) - processed_ids)
        progress["to_process_n"] = len(progress["to_process"])
        
        # Update elapsed time
        progress["end_time"] = datetime.now().isoformat()
        progress["elapsed_seconds"] = progress.get("elapsed_seconds", 0) + round(time.monotonic() - start_ts, 1)
        start_ts = time.monotonic()  # Reset step timer

        # Persist progress to disk
        with open(progress_path, "w") as f:
            json.dump(progress, f, indent=4)
            
        # empty cache before continuing
        gc.collect()
        torch.cuda.empty_cache()

# Handle leftover rows that didn't reach a full 5k chunk
if records:
    chunk_df = pd.DataFrame(records)
    processed_ids = {r["id"] for r in records}
    chunk_df.to_parquet(embeddings_path / "chunks" / "chunk_final.parquet", index=False)
    
    # Final progress update
    progress["ids_processed_n"] += len(processed_ids)
    progress["to_process"] = list(set(progress["to_process"]) - processed_ids)
    progress["to_process_n"] = len(progress["to_process"])
    progress["end_time"] = datetime.now().isoformat()
    
    with open(progress_path, "w") as f:
        json.dump(progress, f, indent=4)
        
    print(f"Finished! Processed all {total_rows:,} items.")


2: 537333
3: 499190
Processed: 1,024 / 499,190
Processed: 2,016 / 499,190
Processed: 3,008 / 499,190
Processed: 4,000 / 499,190
--- Saved Checkpoint: chunk_4000.parquet ---
Processed: 5,024 / 499,190
Processed: 6,016 / 499,190
Processed: 7,008 / 499,190
Processed: 8,000 / 499,190
--- Saved Checkpoint: chunk_8000.parquet ---
Processed: 9,024 / 499,190
Processed: 10,016 / 499,190
Processed: 11,008 / 499,190
Processed: 12,000 / 499,190
--- Saved Checkpoint: chunk_12000.parquet ---
Processed: 13,024 / 499,190
Processed: 14,016 / 499,190
Processed: 15,008 / 499,190
Processed: 16,000 / 499,190
--- Saved Checkpoint: chunk_16000.parquet ---
Processed: 17,024 / 499,190
Processed: 18,016 / 499,190
Processed: 19,008 / 499,190
Processed: 20,000 / 499,190
--- Saved Checkpoint: chunk_20000.parquet ---
Processed: 21,024 / 499,190
Processed: 22,016 / 499,190
Processed: 23,008 / 499,190
Processed: 24,000 / 499,190
--- Saved Checkpoint: chunk_24000.parquet ---
Processed: 25,024 / 499,190
Processed: 26,0